# 27 — Self-RAG

Three **LLM decision gates** that control when and whether to retrieve: classify (should I retrieve?), grade_docs (is each doc relevant?), check_support (is the answer grounded?).

**What you'll learn**
- Self-RAG (Asai et al. 2023): LLM generates reflection tokens to decide retrieval at inference time
- Gate 1 — `classify`: skip retrieval entirely for math, greetings, or simple facts
- Gate 2 — `grade_docs`: filter irrelevant documents before generation
- Gate 3 — `check_support`: verify the answer is grounded in what was actually retrieved
- `BoolDecision(answer: bool)` — one Pydantic model powers all three gates

**Companion examples:** 17-corrective-rag (corrects after bad retrieval), 25-adaptive-rag (routes before retrieval), 29-llm-judge-harness (structured evaluation)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma chromadb python-dotenv langgraph pydantic

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Knowledge base ──────────────────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

DOCUMENTS = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs.", metadata={"source": "langgraph-docs"}),
    Document(page_content="Self-RAG teaches LLMs to generate reflection tokens that decide when to retrieve.", metadata={"source": "self-rag-paper"}),
    Document(page_content="Corrective RAG grades retrieved documents and rewrites the query when docs are irrelevant.", metadata={"source": "crag-paper"}),
    Document(page_content="RAG Fusion generates multiple query variants then merges results via Reciprocal Rank Fusion.", metadata={"source": "rag-fusion"}),
    Document(page_content="The Eiffel Tower is 330 metres tall and stands in Paris, France.", metadata={"source": "general"}),
    Document(page_content="Adaptive RAG classifies each query and routes it to the cheapest correct retrieval path.", metadata={"source": "adaptive-rag"}),
]

store = Chroma.from_documents(DOCUMENTS, OpenAIEmbeddings())
retriever = store.as_retriever(search_kwargs={"k": 2})

print(f"KB ready with {len(DOCUMENTS)} documents")

In [ ]:
# ── 4. BoolDecision + test each gate individually ─────────────────────────────
# One tiny Pydantic model powers all three gates.
# with_structured_output forces the LLM to return {"answer": true/false}.
from langchain_openai import ChatOpenAI
from pydantic import BaseModel


class BoolDecision(BaseModel):
    answer: bool


_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
_gate = _llm.with_structured_output(BoolDecision)

# Try each gate manually before wiring them into the graph
q = "What is Self-RAG?"
sample_doc = DOCUMENTS[1].page_content
sample_answer = "Self-RAG uses reflection tokens to decide whether to retrieve."

g1 = _gate.invoke(f"Should I search a knowledge base to answer: '{q}'? Say false for math or greetings.")
g2 = _gate.invoke(f"Is this doc relevant to '{q}'?\nDoc: {sample_doc}")
g3 = _gate.invoke(f"Is this answer grounded in:\n{sample_doc}\nAnswer: {sample_answer}")

print(f"Gate 1 (should_retrieve) : {g1.answer}")
print(f"Gate 2 (isRelevant)      : {g2.answer}")
print(f"Gate 3 (isSupported)     : {g3.answer}")

In [ ]:
# ── 5. Build the Self-RAG graph ────────────────────────────────────────────────
from typing import TypedDict

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    question: str
    documents: list
    generation: str
    should_retrieve: bool
    supported: bool


def classify(state: State) -> dict:
    """Gate 1 — should we retrieve at all?"""
    result = _gate.invoke(
        f"Should I search a knowledge base to answer: '{state['question']}'? "
        "Say false for math, greetings, or simple facts the model knows well."
    )
    return {"should_retrieve": result.answer}


def retrieve(state: State) -> dict:
    return {"documents": retriever.invoke(state["question"])}


def grade_docs(state: State) -> dict:
    """Gate 2 — keep only relevant documents."""
    relevant = [
        d for d in state["documents"]
        if _gate.invoke(
            f"Is this doc relevant to: '{state['question']}'?\nDoc: {d.page_content}"
        ).answer
    ]
    return {"documents": relevant}


def generate(state: State) -> dict:
    ctx = "\n\n".join(d.page_content for d in state.get("documents", []))
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using the context if provided, otherwise use your knowledge."),
        ("human", "Context:\n{ctx}\n\nQuestion: {question}"),
    ])
    answer = (prompt | _llm | StrOutputParser()).invoke({"ctx": ctx, "question": state["question"]})
    return {"generation": answer}


def check_support(state: State) -> dict:
    """Gate 3 — is the answer grounded in the retrieved context?"""
    if not state.get("documents"):
        return {"supported": True}  # no retrieval → model used its own knowledge
    ctx = "\n\n".join(d.page_content for d in state["documents"])
    result = _gate.invoke(
        f"Is this answer fully supported by the context?\nContext: {ctx}\nAnswer: {state['generation']}"
    )
    return {"supported": result.answer}


g = StateGraph(State)
g.add_node("classify", classify)
g.add_node("retrieve", retrieve)
g.add_node("grade_docs", grade_docs)
g.add_node("generate", generate)
g.add_node("check_support", check_support)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", lambda s: "retrieve" if s.get("should_retrieve") else "generate")
g.add_edge("retrieve", "grade_docs")
g.add_edge("grade_docs", "generate")
g.add_edge("generate", "check_support")
g.add_edge("check_support", END)
app = g.compile()

print("Graph: START → classify → [retrieve → grade_docs →] generate → check_support → END")

In [ ]:
# ── 6. Run Self-RAG demo ───────────────────────────────────────────────────────
QUESTIONS = [
    "What is Self-RAG?",           # retrieval expected
    "What is 17 times 6?",         # no retrieval — math
    "How does Adaptive RAG work?", # retrieval expected
    "Hello, how are you?",         # no retrieval — greeting
]

for q in QUESTIONS:
    result = app.invoke({"question": q, "documents": [], "generation": "", "should_retrieve": False, "supported": False})
    print(f"Q: {q}")
    print(f"  Retrieved={result['should_retrieve']}  Docs kept={len(result['documents'])}  Supported={result['supported']}")
    print(f"  Answer: {result['generation'][:120]}")
    print()

## Exercises

**Exercise 1 — Probe Gate 1:** Ask `"What is the capital of France?"`. The LLM knows Paris without retrieval — does Gate 1 correctly return `False`?

**Exercise 2 — Make Gate 2 filter:** Add an off-topic document to `DOCUMENTS` (e.g., a recipe). Ask a LangGraph question and verify `grade_docs` removes it.

**Exercise 3 — Add a retry loop:** When `check_support` returns `False`, route back to `retrieve` with a rewritten query. How would you wire this conditional edge without creating an infinite loop?

## What's next

- **17-corrective-rag** — corrects AFTER retrieval: grade docs → rewrite query → web search fallback
- **25-adaptive-rag** — routes queries to different retrieval strategies before fetching
- **29-llm-judge-harness** — use structured output to score RAG answers on a 0-3 rubric